# Day 1 - Data Ingestion
**Bluestock Fintech Capstone**
Load all 10 raw datasets, inspect shape/dtypes/head, validate AMFI codes, fetch live NAV.

In [ ]:
import pandas as pd
import requests, time, json
from pathlib import Path

ROOT = Path().resolve().parent
RAW  = ROOT / 'data' / 'raw'

datasets = {
    '01_fund_master':          '01_fund_master.csv',
    '02_nav_history':          '02_nav_history.csv',
    '03_aum_by_fund_house':    '03_aum_by_fund_house.csv',
    '04_monthly_sip_inflows':  '04_monthly_sip_inflows.csv',
    '05_category_inflows':     '05_category_inflows.csv',
    '06_industry_folio_count': '06_industry_folio_count.csv',
    '07_scheme_performance':   '07_scheme_performance.csv',
    '08_investor_transactions':'08_investor_transactions.csv',
    '09_portfolio_holdings':   '09_portfolio_holdings.csv',
    '10_benchmark_indices':    '10_benchmark_indices.csv',
}
loaded = {}
for name, fname in datasets.items():
    df = pd.read_csv(RAW / fname)
    loaded[name] = df
    print(f"{name}: {df.shape} | nulls={df.isnull().sum().sum()}")


In [ ]:
# inspect each dataset
for name, df in loaded.items():
    print(f'\n=== {name} ===')
    print(f'Shape : {df.shape}')
    print(f'Cols  : {list(df.columns)}')
    print(df.head(2).to_string())


In [ ]:
# validate AMFI codes between fund_master and nav_history
master_codes = set(loaded['01_fund_master']['amfi_code'].unique())
nav_codes    = set(loaded['02_nav_history']['amfi_code'].unique())
print(f'Codes in fund_master : {len(master_codes)}')
print(f'Codes in nav_history : {len(nav_codes)}')
print(f'Missing in nav       : {master_codes - nav_codes}')
print(f'Missing in master    : {nav_codes - master_codes}')


In [ ]:
# explore fund master - unique categories
fm = loaded['01_fund_master']
for col in ['fund_house','category','sub_category','risk_category','plan']:
    vals = sorted(fm[col].unique())
    print(f'{col} ({len(vals)}): {vals}')


In [ ]:
# live NAV fetch from mfapi.in
SCHEMES = {
    125497: ('HDFC Top 100 Direct',     'live_hdfc_top100.csv'),
    119551: ('SBI Bluechip Regular',    'live_sbi_bluechip.csv'),
    120503: ('ICICI Pru Bluechip',      'live_icici_bluechip.csv'),
    118632: ('Nippon Large Cap',        'live_nippon_largecap.csv'),
    119092: ('Axis Bluechip Regular',   'live_axis_bluechip.csv'),
    120841: ('Kotak Bluechip Regular',  'live_kotak_bluechip.csv'),
}
HEADERS = {'User-Agent': 'Mozilla/5.0 Chrome/120', 'Accept': 'application/json'}

for code, (name, fname) in SCHEMES.items():
    try:
        r = requests.get(f'https://api.mfapi.in/mf/{code}', headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
        if data.get('status') == 'SUCCESS':
            df = pd.DataFrame(data['data'])
            df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
            df['nav']  = pd.to_numeric(df['nav'])
            df['amfi_code'] = code
            df = df.sort_values('date').reset_index(drop=True)
            df.to_csv(RAW / fname, index=False)
            print(f'Fetched {name}: {len(df)} rows | {df["date"].min().date()} to {df["date"].max().date()}')
        else:
            print(f'API error for {name}')
    except Exception as e:
        print(f'Failed {name}: {e}')
    time.sleep(1)
